In [ ]:
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np

# Add src folder to path so we can import our modules
sys.path.append('../src')

from tokenizer import LatexTokenizer
from augmentations import get_training_transforms
from dataset import get_dataloader

# --- 1. Mock Data (To test without downloading all of CROHME) ---
import os
from PIL import Image

os.makedirs('../data/raw', exist_ok=True)
dummy_img_path = '../data/raw/dummy_1.png'
# Create a dummy white image to act as paper
Image.fromarray(np.ones((256, 256), dtype=np.uint8) * 255).save(dummy_img_path)

mock_gt = {'dummy_1.png': '\\int _ { 0 } ^ { \\infty } e ^ { - x } d x'}

# --- 2. Initialize Phase 1 Pipeline ---
tokenizer = LatexTokenizer()
tokenizer.fit_on_texts(list(mock_gt.values()))

transforms = get_training_transforms(image_size=(256, 256))

loader = get_dataloader(
    data_dir='../data/raw', 
    gt_dict=mock_gt, 
    tokenizer=tokenizer, 
    transforms=transforms, 
    batch_size=1
)

# --- 3. Fetch and Visualize ---
images, targets = next(iter(loader))

img_to_show = images[0].squeeze().numpy()
target_tokens = targets[0]

print("--- Pipeline Outputs ---")
print(f"Image Tensor Shape: {images.shape}")
print(f"Target Tensor Shape: {targets.shape}")
print(f"Decoded Target: {tokenizer.decode(target_tokens)}")

plt.figure(figsize=(6, 6))
plt.imshow(img_to_show, cmap='gray')
plt.title("Warped Image (Fed to Neural Net)")
plt.axis('off')
plt.show()